In [1]:
"""
AnalystLab Africa - Track C: Machine Learning
Notebook: pipeline_ml.ipynb
Step 1: Import required libraries
"""

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Model selection and preprocessing pipelines
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Machine Learning algorithms
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries imported successfully.")

All libraries imported successfully.


In [5]:
"""
Step 2: Load raw dataset from the data folder and inspect the first records
"""

# Relative path from 'notebooks/' to 'data/Listings.csv'
DATA_PATH = os.path.join("..", "data", "Listings.csv")

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"File not found: {DATA_PATH}. Check your data folder structure.")

print(f"Loading raw dataset from {DATA_PATH}...")

# Specify encoding="latin1" to handle special characters and accents
df_raw = pd.read_csv(DATA_PATH, encoding="latin1", low_memory=False)

print(f"Dataset successfully loaded.")
print(f"Dimensions: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns.\n")


# Display the first 5 rows
df_raw.head(5)

Loading raw dataset from ..\data\Listings.csv...
Dataset successfully loaded.
Dimensions: 279,712 rows and 33 columns.



,listing_id,name,host_id,host_since,host_location,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,...,minimum_nights,maximum_nights,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable
0,281420,"Beautiful Flat in le Village Montmartre, Paris",1466919,2011-12-03,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
1,3705183,39 mÃÂ² Paris (Sacre CÃ âur),10328771,2013-11-29,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
2,4082273,"Lovely apartment with Terrace, 60m2",19252768,2014-07-31,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
3,4797344,Cosy studio (close to Eiffel tower),10668311,2013-12-17,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f
4,4823489,Close to Eiffel Tower - Beautiful flat : 2 rooms,24837558,2014-12-14,"Paris, Ile-de-France, France",NaN,NaN,NaN,f,1.0,...,2,1125,100.0,10.0,10.0,10.0,10.0,10.0,10.0,f


In [6]:
"""
Step 3: Scope by city to standardize currency
Currency limitation: listings lack currency tags, so prices represent local values.
Scoping strictly to Paris ensures uniform analysis in Euros (EUR).
"""

df_paris = df_raw[df_raw["city"] == "Paris"].copy()
print(f"Total listings extracted for Paris: {len(df_paris):,}")

Total listings extracted for Paris: 64,690


In [7]:
"""
Step 4: Clean the target variable (price) and trim extreme outliers
"""

# Clean formatting symbols if price was loaded as text
if df_paris["price"].dtype == object:
    df_paris["price"] = df_paris["price"].astype(str).str.replace(r"[\$,]", "", regex=True)

df_paris["price"] = pd.to_numeric(df_paris["price"], errors="coerce")

# Filter out non-positive, zero, or missing prices
df_paris = df_paris[df_paris["price"] > 0]

# Remove extreme outliers (1st to 99th percentile) to mitigate heavy-tail distortion
p_low = df_paris["price"].quantile(0.01)
p_high = df_paris["price"].quantile(0.99)

df_clean = df_paris[(df_paris["price"] >= p_low) & (df_paris["price"] <= p_high)].copy()

print(f"Price range retained: €{p_low:.0f} to €{p_high:.0f}")
print(f"Listings retained after price bounds: {len(df_clean):,}")

Price range retained: €25 to €600
Listings retained after price bounds: 63,520


In [8]:
"""
Step 5: Feature selection and missing value inspection
"""

numeric_features = [
    "accommodates",
    "bedrooms",
    "bathrooms",
    "review_scores_rating"
]

categorical_features = [
    "room_type",
    "property_type",
    "host_is_superhost"
]

# Keep only existing columns from the dataframe
available_num = [col for col in numeric_features if col in df_clean.columns]
available_cat = [col for col in categorical_features if col in df_clean.columns]

X = df_clean[available_num + available_cat].copy()
y = df_clean["price"].copy()

print("Missing values count per selected feature:")
print(X.isnull().sum())

Missing values count per selected feature:
accommodates                0
bedrooms                13192
review_scores_rating    16120
room_type                   0
property_type               0
host_is_superhost          33
dtype: int64


In [9]:
"""
Step 6: Train / Test Split (Prevents data leakage before imputation)
"""

# Split 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Train samples: {len(X_train):,} | Test samples: {len(X_test):,}")

# Statistical summary computed exclusively on training features
print("\nDescriptive statistics of training numeric features:")
X_train[available_num].describe().round(2)

Train samples: 50,816 | Test samples: 12,704

Descriptive statistics of training numeric features:


,accommodates,bedrooms,review_scores_rating
count,50816.00,40241.00,37950.00
mean,3.02,1.36,93.06
std,1.51,0.99,8.89
min,1.00,1.00,20.00
25%,2.00,1.00,90.00
50%,2.00,1.00,95.00
75%,4.00,2.00,100.00
max,16.00,50.00,100.00


In [10]:
"""
Step 7: Preprocessing pipelines
- Numerical: median imputation + standard scaling (fit on train only)
- Categorical: mode imputation + one-hot encoding (fit on train only)
"""

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, available_num),
        ("cat", cat_transformer, available_cat)
    ]
)

print("Preprocessing pipeline built successfully.")

Preprocessing pipeline built successfully.


In [11]:
"""
Step 8: Model training, evaluation, and benchmark comparison
"""

# Evaluation helper function
def evaluate_model(name, pipeline):
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    return {
        "Model": name,
        "MAE (EUR)": round(mae, 2),
        "RMSE (EUR)": round(rmse, 2),
        "R2 Score": round(r2, 4)
    }

results = []

# 1. Baseline Model (Median Regressor)
print("Training Baseline Model...")
baseline_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", DummyRegressor(strategy="median"))
])
baseline_pipe.fit(X_train, y_train)
results.append(evaluate_model("Baseline (Median)", baseline_pipe))

# 2. Linear Model (Ridge Regression)
print("Training Ridge Regression...")
ridge_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", Ridge(alpha=1.0))
])
ridge_pipe.fit(X_train, y_train)
results.append(evaluate_model("Ridge Regression", ridge_pipe))

# 3. Non-linear Model (Random Forest Regressor)
print("Training Random Forest Regressor...")
rf_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1))
])
rf_pipe.fit(X_train, y_train)
results.append(evaluate_model("Random Forest", rf_pipe))

# Display benchmark table
results_df = pd.DataFrame(results)
print("\n" + "=" * 48)
print("             BENCHMARK RESULTS")
print("=" * 48)
results_df

Training Baseline Model...
Training Ridge Regression...
Training Random Forest Regressor...

             BENCHMARK RESULTS


,Model,MAE (EUR),RMSE (EUR),R2 Score
0,Baseline (Median),45.79,79.04,-0.0911
1,Ridge Regression,37.89,60.00,0.3713
2,Random Forest,36.16,58.04,0.4116


In [12]:
"""
Step 9: Export the winning model pipeline to the models folder
"""

# Ensure the output directory exists
os.makedirs(os.path.join("..", "models"), exist_ok=True)

MODEL_OUTPUT_PATH = os.path.join("..", "models", "best_airbnb_model.joblib")

# Save the complete pipeline (preprocessor + fitted Random Forest)
joblib.dump(rf_pipe, MODEL_OUTPUT_PATH)
print(f"Model pipeline successfully saved to '{MODEL_OUTPUT_PATH}'")

Model pipeline successfully saved to '..\models\best_airbnb_model.joblib'
